<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Chapter 3: Coding Attention Mechanisms

Packages that are being used in this notebook:

In [4]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.12.0


- This chapter covers attention mechanisms, the engine of LLMs:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/01.webp?123" width="500px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/02.webp" width="600px">

## 3.1 The problem with modeling long sequences

- No code in this section
- Translating a text word by word isn't feasible due to the differences in grammatical structures between the source and target languages:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/03.webp" width="400px">

- Prior to the introduction of transformer models, encoder-decoder RNNs were commonly used for machine translation tasks
- In this setup, the encoder processes a sequence of tokens from the source language, using a hidden state—a kind of intermediate layer within the neural network—to generate a condensed representation of the entire input sequence:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/04.webp" width="500px">

## 3.2 Capturing data dependencies with attention mechanisms

- No code in this section
- Through an attention mechanism, the text-generating decoder segment of the network is capable of selectively accessing all input tokens, implying that certain input tokens hold more significance than others in the generation of a specific output token:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/05.webp" width="500px">

- Self-attention in transformers is a technique designed to enhance input representations by enabling each position in a sequence to engage with and determine the relevance of every other position within the same sequence

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/06.webp" width="300px">

## 3.3 Attending to different parts of the input with self-attention

### 3.3.1 A simple self-attention mechanism without trainable weights

- This section explains a very simplified variant of self-attention, which does not contain any trainable weights
- This is purely for illustration purposes and NOT the attention mechanism that is used in transformers
- The next section, section 3.3.2, will extend this simple attention mechanism to implement the real self-attention mechanism
- Suppose we are given an input sequence $x^{(1)}$ to $x^{(T)}$
  - The input is a text (for example, a sentence like "Your journey starts with one step") that has already been converted into token embeddings as described in chapter 2
  - For instance, $x^{(1)}$ is a d-dimensional vector representing the word "Your", and so forth
- **Goal:** compute context vectors $z^{(i)}$ for each input sequence element $x^{(i)}$ in $x^{(1)}$ to $x^{(T)}$ (where $z$ and $x$ have the same dimension)
    - A context vector $z^{(i)}$ is a weighted sum over the inputs $x^{(1)}$ to $x^{(T)}$
    - The context vector is "context"-specific to a certain input
      - Instead of $x^{(i)}$ as a placeholder for an arbitrary input token, let's consider the second input, $x^{(2)}$
      - And to continue with a concrete example, instead of the placeholder $z^{(i)}$, we consider the second output context vector, $z^{(2)}$
      - The second context vector, $z^{(2)}$, is a weighted sum over all inputs $x^{(1)}$ to $x^{(T)}$ weighted with respect to the second input element, $x^{(2)}$
      - The attention weights are the weights that determine how much each of the input elements contributes to the weighted sum when computing $z^{(2)}$
      - In short, think of $z^{(2)}$ as a modified version of $x^{(2)}$ that also incorporates information about all other input elements that are relevant to a given task at hand

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/07.webp" width="400px">

- (Please note that the numbers in this figure are truncated to one
digit after the decimal point to reduce visual clutter; similarly, other figures may also contain truncated values)

- By convention, the unnormalized attention weights are referred to as **"attention scores"** whereas the normalized attention scores, which sum to 1, are referred to as **"attention weights"**


- The code below walks through the figure above step by step

<br>

- **Step 1:** compute unnormalized attention scores $\omega$
- Suppose we use the second input token as the query, that is, $q^{(2)} = x^{(2)}$, we compute the unnormalized attention scores via dot products:
    - $\omega_{21} = x^{(1)} q^{(2)\top}$
    - $\omega_{22} = x^{(2)} q^{(2)\top}$
    - $\omega_{23} = x^{(3)} q^{(2)\top}$
    - ...
    - $\omega_{2T} = x^{(T)} q^{(2)\top}$
- Above, $\omega$ is the Greek letter "omega" used to symbolize the unnormalized attention scores
    - The subscript "21" in $\omega_{21}$ means that input sequence element 2 was used as a query against input sequence element 1

- Suppose we have the following input sentence that is already embedded in 3-dimensional vectors as described in chapter 3 (we use a very small embedding dimension here for illustration purposes, so that it fits onto the page without line breaks):

In [6]:
#准备输入数据
import torch
# 导入 PyTorch 库。
# 后面要使用 torch.tensor、torch.dot、torch.softmax、矩阵乘法等功能。

inputs = torch.tensor(
    # 创建一个 PyTorch 张量，用来表示输入句子中所有词元的嵌入向量。
    # 这个张量的形状是 [6, 3]：
    # 6 表示句子中有 6 个词元；
    # 3 表示每个词元用 3 个数字表示，也就是 3 维嵌入向量。

    [[0.43, 0.15, 0.89],  # Your    (x^1)
     # 第 1 个词元 "Your" 的嵌入向量。
     # 可以理解为模型用这 3 个数字表示 "Your" 这个词。

     [0.55, 0.87, 0.66],  # journey (x^2)
     # 第 2 个词元 "journey" 的嵌入向量。
     # 后面我们会把它作为 query，也就是重点研究对象。

     [0.57, 0.85, 0.64],  # starts  (x^3)
     # 第 3 个词元 "starts" 的嵌入向量。

     [0.22, 0.58, 0.33],  # with    (x^4)
     # 第 4 个词元 "with" 的嵌入向量。

     [0.77, 0.25, 0.10],  # one     (x^5)
     # 第 5 个词元 "one" 的嵌入向量。

     [0.05, 0.80, 0.55]]  # step    (x^6)
     # 第 6 个词元 "step" 的嵌入向量。
)
#把一句话表示成一个二维矩阵：
#每一行是一个词，每一列是这个词向量中的一个维度。

print(inputs)
print(inputs.shape)

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
torch.Size([6, 3])


- (In this book, we follow the common machine learning and deep learning convention where training examples are represented as rows and feature values as columns; in the case of the tensor shown above, each row represents a word, and each column represents an embedding dimension)

- The primary objective of this section is to demonstrate how the context vector $z^{(2)}$
  is calculated using the second input sequence, $x^{(2)}$, as a query

- The figure depicts the initial step in this process, which involves calculating the attention scores ω between $x^{(2)}$
  and all other input elements through a dot product operation

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/08.webp" width="400px">

- We use input sequence element 2, $x^{(2)}$, as an example to compute context vector $z^{(2)}$; later in this section, we will generalize this to compute all context vectors.
- The first step is to compute the unnormalized attention scores by computing the dot product between the query $x^{(2)}$ and all other input tokens:

In [10]:
#计算第二个词元 journey 的注意力分数
query = inputs[1]

# 取出第 2 个词元 "journey" 的向量。
# Python 下标从 0 开始，所以 inputs[1] 表示第 2 行。
# 这里把 "journey" 作为 query，也就是当前要计算上下文向量的目标词元。

print(query)

attn_scores_2 = torch.empty(inputs.shape[0])
# 创建一个空的一维张量，用来保存 "journey" 和每个输入词元之间的注意力分数。
# inputs.shape[0] 等于 6，表示输入序列中有 6 个词元。
# 所以 attn_scores_2 的形状是 [6]。
# 后面每个位置会存一个分数：
# 第 1 个分数：journey 和 Your 的相关性；
# 第 2 个分数：journey 和 journey 自己的相关性；
# 第 3 个分数：journey 和 starts 的相关性；
# 以此类推。

for i, x_i in enumerate(inputs):
    # 遍历 inputs 中的每一个词元向量。
    # i 是当前词元的下标，x_i 是当前词元对应的 3 维向量。

    attn_scores_2[i] = torch.dot(x_i, query)
    # 计算当前词元 x_i 和 query，也就是 "journey" 向量之间的点积。
    # torch.dot(a, b) 会把两个一维向量对应位置相乘，然后求和。
    # 点积结果是一个标量，表示两个向量之间的相似程度。
    # 这个分数被保存到 attn_scores_2[i] 中。

print(attn_scores_2)
# 打印 "journey" 对所有词元的注意力分数。

tensor([0.5500, 0.8700, 0.6600])
tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


- Side note: a dot product is essentially a shorthand for multiplying two vectors elements-wise and summing the resulting products:

In [11]:
#手动理解点积计算
res = 0.
# 初始化一个变量 res，用来保存点积计算结果。
# 这里先设为 0。

for idx, element in enumerate(inputs[0]):
    # 遍历第 1 个词元 "Your" 的向量。
    # idx 是当前维度的下标；
    # element 是当前维度上的值。
    # 注意：这里 element 实际上没有直接使用，代码通过 inputs[0][idx] 取值。

    res += inputs[0][idx] * query[idx]
    # 把 "Your" 向量和 "journey" 向量在同一个维度上的值相乘。
    # 然后把乘积累加到 res 中。
    #
    # 具体来说：
    # inputs[0] 是 Your 的向量：[0.43, 0.15, 0.89]
    # query 是 journey 的向量：[0.55, 0.87, 0.66]
    #
    # 点积 = 0.43*0.55 + 0.15*0.87 + 0.89*0.66

print(res)
# 打印手动计算出来的点积结果。

print(torch.dot(inputs[0], query))
# 使用 PyTorch 内置的 torch.dot() 计算同样的点积。
# 这行代码是为了验证：手动计算结果和 torch.dot() 的结果一致。

tensor(0.9544)
tensor(0.9544)


- **Step 2:** normalize the unnormalized attention scores ("omegas", $\omega$) so that they sum up to 1
- Here is a simple way to normalize the unnormalized attention scores to sum up to 1 (a convention, useful for interpretation, and important for training stability):

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/09.webp" width="500px">

In [12]:
#把注意力分数归一化成注意力权重：简单除法版本
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
# 把每个注意力分数除以所有注意力分数的总和。
# 这样处理后，所有值加起来就等于 1。
#
# 例如：
# 原始分数是 [0.9544, 1.4950, ...]
# 分数总和是所有分数加起来。
# 每个分数 / 总和，就得到一个比例。
#
# 这个比例就可以理解为：
# "journey" 应该分别关注每个词元多少。

print("Attention weights:", attn_weights_2_tmp)
# 打印归一化后的注意力权重。

print("Sum:", attn_weights_2_tmp.sum())
# 打印所有注意力权重的总和。
# 正常情况下应该接近 1。

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


- However, in practice, using the softmax function for normalization, which is better at handling extreme values and has more desirable gradient properties during training, is common and recommended.
- Here's a naive implementation of a softmax function for scaling, which also normalizes the vector elements such that they sum up to 1:

In [13]:
#手写一个简单版 softmax
def softmax_naive(x):
    # 定义一个简单版本的 softmax 函数。
    # 输入 x 是一个张量，里面保存的是注意力分数。

    return torch.exp(x) / torch.exp(x).sum(dim=0)
    # torch.exp(x) 会对 x 中每个元素做指数运算。
    # 指数运算会把所有值变成正数，并且会放大较大分数之间的差异。
    #
    # torch.exp(x).sum(dim=0) 计算所有指数值的总和。
    #
    # 每个指数值 / 指数总和，就得到 softmax 权重。
    # 这样得到的所有权重加起来等于 1。

attn_weights_2_naive = softmax_naive(attn_scores_2)
# 使用手写的 softmax_naive 函数，把注意力分数转换成注意力权重。

print("Attention weights:", attn_weights_2_naive)
# 打印 softmax 得到的注意力权重。

print("Sum:", attn_weights_2_naive.sum())
# 打印注意力权重总和。
# 应该等于 1。

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- The naive implementation above can suffer from numerical instability issues for large or small input values due to overflow and underflow issues
- Hence, in practice, it's recommended to use the PyTorch implementation of softmax instead, which has been highly optimized for performance:

In [17]:
#使用 PyTorch 官方 softmax
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
# 使用 PyTorch 官方的 softmax 函数。
# attn_scores_2 是前面计算出来的一维注意力分数张量。
#
# dim=0 表示沿着第 0 个维度做 softmax。
# 因为 attn_scores_2 是一维张量，所以 dim=0 就是对整个向量做 softmax。
#
# 结果 attn_weights_2 仍然是一个长度为 6 的一维张量。
# 里面每个值表示 "journey" 对某个词元的关注比例。

print("Attention weights:", attn_weights_2)
# 打印注意力权重。

print("Sum:", attn_weights_2.sum())
# 打印注意力权重总和，应该是 1。

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- **Step 3**: compute the context vector $z^{(2)}$ by multiplying the embedded input tokens, $x^{(i)}$ with the attention weights and sum the resulting vectors:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/10.webp" width="500px">

In [19]:
#计算第二个词元 journey 的上下文向量
query = inputs[1]
print(query)
# 再次取出第 2 个词元 "journey" 的向量。
# 这里 query 本身不是必须重新赋值，但书中为了清晰又写了一遍。

context_vec_2 = torch.zeros(query.shape)
# 创建一个和 query 形状相同的零向量。
# query.shape 是 [3]，所以 context_vec_2 也是一个 3 维向量。
# 后面会把所有加权后的输入向量累加到这个变量上。
#
# 可以理解为：
# 先准备一个空篮子，后面把各个词元的信息按权重放进去。

for i, x_i in enumerate(inputs):
    # 遍历所有输入词元。
    # i 是词元下标；
    # x_i 是当前词元的 3 维嵌入向量。

    context_vec_2 += attn_weights_2[i] * x_i
    # 用第 i 个注意力权重乘以第 i 个输入向量。
    # 然后把结果累加到 context_vec_2 中。
    #
    # 权重大的词元，对最终上下文向量贡献更大；
    # 权重小的词元，对最终上下文向量贡献更小。
    #
    # 这一步就是加权求和：
    # context_vec_2 =
    #   权重1 * Your向量
    # + 权重2 * journey向量
    # + 权重3 * starts向量
    # + ...
    # + 权重6 * step向量

print(context_vec_2)
# 打印最终得到的第 2 个词元 "journey" 的上下文向量。

tensor([0.5500, 0.8700, 0.6600])
tensor([0.4419, 0.6515, 0.5683])


### 3.3.2 Computing attention weights for all input tokens

#### Generalize to all input sequence tokens:

- Above, we computed the attention weights and context vector for input 2 (as illustrated in the highlighted row in the figure below)
- Next, we are generalizing this computation to compute all attention weights and context vectors

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/11.webp" width="400px">

- (Please note that the numbers in this figure are truncated to two
digits after the decimal point to reduce visual clutter; the values in each row should add up to 1.0 or 100%; similarly, digits in other figures are truncated)

- In self-attention, the process starts with the calculation of attention scores, which are subsequently normalized to derive attention weights that total 1
- These attention weights are then utilized to generate the context vectors through a weighted summation of the inputs

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/12.webp" width="400px">

- Apply previous **step 1** to all pairwise elements to compute the unnormalized attention score matrix:

In [20]:
#计算所有词元之间的注意力分数：for 循环版本
attn_scores = torch.empty(6, 6)
print(attn_scores)
# 创建一个 6 行 6 列的空张量，用来保存所有词元两两之间的注意力分数。
#
# 为什么是 6x6？
# 因为输入句子有 6 个词元。
# 每个词元都要和另外 6 个词元分别计算相关性。
#
# 第 i 行第 j 列表示：
# 第 i 个词元 对 第 j 个词元 的注意力分数。

for i, x_i in enumerate(inputs):
    # 外层循环：遍历每一个词元，作为当前 query。
    # i 是 query 词元的下标；
    # x_i 是 query 词元的向量。

    for j, x_j in enumerate(inputs):
        # 内层循环：遍历每一个词元，作为被关注的对象。
        # j 是被关注词元的下标；
        # x_j 是被关注词元的向量。

        attn_scores[i, j] = torch.dot(x_i, x_j)
        # 计算第 i 个词元和第 j 个词元之间的点积。
        # 点积结果保存到 attn_scores[i, j]。
        #
        # 例如：
        # attn_scores[1, 2] 表示 journey 和 starts 的注意力分数。

print(attn_scores)
# 打印完整的 6x6 注意力分数矩阵。

tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- We can achieve the same as above more efficiently via matrix multiplication:

In [23]:
attn_scores = torch.empty(6, 6)

#用矩阵乘法计算所有注意力分数
print(attn_scores)
attn_scores = inputs @ inputs.T
# 使用矩阵乘法一次性计算所有词元两两之间的点积。
#
# inputs 的形状是 [6, 3]。
# inputs.T 是 inputs 的转置，形状是 [3, 6]。
#
# [6, 3] @ [3, 6] = [6, 6]
#
# 得到的结果就是 6x6 的注意力分数矩阵。
#
# 这和前面双重 for 循环计算出来的结果是一样的，
# 但是矩阵乘法更简洁、更快，也更适合 GPU 加速。

print(attn_scores)
# 打印注意力分数矩阵。

tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- Similar to **step 2** previously, we normalize each row so that the values in each row sum to 1:

In [24]:
#对所有行做 softmax，得到所有词元的注意力权重
attn_weights = torch.softmax(attn_scores, dim=-1)
# 对注意力分数矩阵做 softmax，得到注意力权重矩阵。
#
# attn_scores 的形状是 [6, 6]。
# dim=-1 表示沿着最后一个维度做 softmax。
#
# 对二维矩阵来说，dim=-1 等价于对每一行做 softmax。
#
# 也就是说：
# 第 1 行变成 Your 对所有词元的注意力权重；
# 第 2 行变成 journey 对所有词元的注意力权重；
# 第 3 行变成 starts 对所有词元的注意力权重；
# ...
#
# 每一行的 6 个权重加起来都等于 1。

print(attn_weights)
# 打印注意力权重矩阵。

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


- Quick verification that the values in each row indeed sum to 1:

In [25]:
#验证注意力权重每一行的和是否为 1
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
# 手动把第 2 行的注意力权重加起来。
# 第 2 行对应的是 "journey" 对所有词元的注意力权重。
# 理论上结果应该是 1。

print("Row 2 sum:", row_2_sum)
# 打印第 2 行权重之和。

print("All row sums:", attn_weights.sum(dim=-1))
# 对 attn_weights 的每一行求和。
# dim=-1 表示沿最后一个维度求和。
# 对二维矩阵来说，就是每一行的 6 个数相加。
#
# 如果 softmax 正确，每一行的和都应该是 1。

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


- Apply previous **step 3** to compute all context vectors:

In [26]:
#一次性计算所有词元的上下文向量
print(attn_weights)
print(inputs)
all_context_vecs = attn_weights @ inputs
# 用注意力权重矩阵乘以输入向量矩阵。
#
# attn_weights 的形状是 [6, 6]。
# inputs 的形状是 [6, 3]。
#
# [6, 6] @ [6, 3] = [6, 3]
#
# 得到的 all_context_vecs 形状仍然是 [6, 3]。
#
# 其中：
# 第 1 行是 Your 的上下文向量；
# 第 2 行是 journey 的上下文向量；
# 第 3 行是 starts 的上下文向量；
# ...
# 第 6 行是 step 的上下文向量。
#
# 每个上下文向量都是所有输入词元向量的加权和。

print(all_context_vecs)
# 打印所有词元的上下文向量。

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


- As a sanity check, the previously computed context vector $z^{(2)} = [0.4419, 0.6515, 0.5683]$ can be found in the 2nd row in above: 

In [27]:
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


## 3.4 Implementing self-attention with trainable weights

- A conceptual framework illustrating how the self-attention mechanism developed in this section integrates into the overall narrative and structure of this book and chapter

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/13.webp" width="400px">

### 3.4.1 Computing the attention weights step by step

- In this section, we are implementing the self-attention mechanism that is used in the original transformer architecture, the GPT models, and most other popular LLMs
- This self-attention mechanism is also called "scaled dot-product attention"
- The overall idea is similar to before:
  - We want to compute context vectors as weighted sums over the input vectors specific to a certain input element
  - For the above, we need attention weights
- As you will see, there are only slight differences compared to the basic attention mechanism introduced earlier:
  - The most notable difference is the introduction of weight matrices that are updated during model training
  - These trainable weight matrices are crucial so that the model (specifically, the attention module inside the model) can learn to produce "good" context vectors

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/14.webp" width="600px">

- Implementing the self-attention mechanism step by step, we will start by introducing the three training weight matrices $W_q$, $W_k$, and $W_v$
- These three matrices are used to project the embedded input tokens, $x^{(i)}$, into query, key, and value vectors via matrix multiplication:

  - Query vector: $q^{(i)} = W_q \,x^{(i)}$
  - Key vector: $k^{(i)} = W_k \,x^{(i)}$
  - Value vector: $v^{(i)} = W_v \,x^{(i)}$


- The embedding dimensions of the input $x$ and the query vector $q$ can be the same or different, depending on the model's design and specific implementation
- In GPT models, the input and output dimensions are usually the same, but for illustration purposes, to better follow the computation, we choose different input and output dimensions here:

In [28]:
#定义输入维度和输出维度
x_2 = inputs[1] # second input element
d_in = inputs.shape[1] # the input embedding size, d=3
d_out = 2 # the output embedding size, d=2

print(x_2)
print(d_in)
print(d_out)

tensor([0.5500, 0.8700, 0.6600])
3
2


- Below, we initialize the three weight matrices; note that we are setting `requires_grad=False` to reduce clutter in the outputs for illustration purposes, but if we were to use the weight matrices for model training, we would set `requires_grad=True` to update these matrices during model training

In [29]:
#初始化 Query、Key、Value 三个权重矩阵
#设置随机种子。作用是让每次运行时生成的随机数相同，方便复现实验结果。
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

#W_query：让当前词变成“我要找什么信息”
#W_key：让每个词变成“我有什么索引标签”
#W_value：让每个词变成“我真正提供什么内容”
print(W_query)
print(W_key)
print(W_value)

Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])
Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])
Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


- Next we compute the query, key, and value vectors:

In [30]:
#用第二个词元向量 x_2 乘以 W_query，得到第二个词元的 Query 向量。
query_2 = x_2 @ W_query # _2 because it's with respect to the 2nd input element
#用第二个词元向量乘以 W_key，得到第二个词元的 Key 向量。
key_2 = x_2 @ W_key
#用第二个词元向量乘以 W_value，得到第二个词元的 Value 向量。
value_2 = x_2 @ W_value

print(query_2)
print(key_2)
print(value_2)

tensor([0.4306, 1.4551])
tensor([0.4433, 1.1419])
tensor([0.3951, 1.0037])


- As we can see below, we successfully projected the 6 input tokens from a 3D onto a 2D embedding space:

In [31]:

#计算所有词元的 Key 和 Value
keys = inputs @ W_key 
#把所有输入词元都乘以 W_value，得到所有词元的 Value 向量。
values = inputs @ W_value

#inputs: [6, 3]
#W_value:[3, 2]
#values: [6, 2]

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)
print(keys)
print(values)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])
tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])
tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])


- In the next step, **step 2**, we compute the unnormalized attention scores by computing the dot product between the query and each key vector:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/15.webp" width="600px">

In [34]:
#计算第二个词元和自身 Key 的注意力分数
#取出第二个词元对应的 Key 向量。
keys_2 = keys[1] # Python starts index at 0
print(key_2)
print(query_2)
#计算第二个词元的 Query 向量和第二个词元的 Key 向量之间的点积。这个点积就是一个注意力分数。
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor([0.4433, 1.1419])
tensor([0.4306, 1.4551])
tensor(1.8524)


- Since we have 6 inputs, we have 6 attention scores for the given query vector:

In [35]:
#计算第二个词元对所有词元的注意力分数
#用第二个词元的 Query 向量 query_2，和所有词元的 Key 向量做点积。
#query_2: [2]
#keys.T:  [2, 6]
#结果:    [6]
attn_scores_2 = query_2 @ keys.T # All attention scores for given query
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/16.webp" width="600px">

- Next, in **step 3**, we compute the attention weights (normalized attention scores that sum up to 1) using the softmax function we used earlier
- The difference to earlier is that we now scale the attention scores by dividing them by the square root of the embedding dimension, $\sqrt{d_k}$ (i.e., `d_k**0.5`):

In [45]:
#对注意力分数进行缩放和 softmax 归一化
d_k = keys.shape[1]
print(keys)
print(d_k)
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]])
2
tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/17.webp" width="600px">

- In **step 4**, we now compute the context vector for input query vector 2:

In [43]:
#根据注意力权重加权 Value，得到上下文向量
#attn_weights_2: [6]
#values:         [6, 2]
#context_vec_2:  [2]
#第二个词元的新表示 = 所有词元 Value 的加权和
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


### 3.4.2 Implementing a compact SelfAttention class

- Putting it all together, we can implement the self-attention mechanism as follows:

In [49]:
#输入 x
#   ↓
#计算 keys、queries、values
#   ↓
#queries @ keys.T 得到注意力分数
#   ↓
#softmax 得到注意力权重
#   ↓
#注意力权重 @ values 得到上下文向量
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/18.webp" width="400px">

- We can streamline the implementation above using PyTorch's Linear layers, which are equivalent to a matrix multiplication if we disable the bias units
- Another big advantage of using `nn.Linear` over our manual `nn.Parameter(torch.rand(...)` approach is that `nn.Linear` has a preferred weight initialization scheme, which leads to more stable model training

In [53]:
#SelfAttention_v2 和 SelfAttention_v1 的计算逻辑一样。
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


- Note that `SelfAttention_v1` and `SelfAttention_v2` give different outputs because they use different initial weights for the weight matrices

## 3.5 Hiding future words with causal attention

- In causal attention, the attention weights above the diagonal are masked, ensuring that for any given input, the LLM is unable to utilize future tokens while calculating the context vectors with the attention weight

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/19.webp" width="400px">

### 3.5.1 Applying a causal attention mask

- In this section, we are converting the previous self-attention mechanism into a causal self-attention mechanism
- Causal self-attention ensures that the model's prediction for a certain position in a sequence is only dependent on the known outputs at previous positions, not on future positions
- In simpler words, this ensures that each next word prediction should only depend on the preceding words
- To achieve this, for each given token, we mask out the future tokens (the ones that come after the current token in the input text):

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="600px">

- To illustrate and implement causal self-attention, let's work with the attention scores and weights from the previous section: 

In [65]:
#先计算普通注意力权重
# Reuse the query and key weight matrices of the
# SelfAttention_v2 object from the previous section for convenience
# 使用前面 3.4.2 节中已经定义好的 SelfAttention_v2 对象 sa_v2
# W_query 是查询向量的线性变换层
# inputs 的形状大致是 [num_tokens, d_in]
# queries 的形状是 [num_tokens, d_out]
queries = sa_v2.W_query(inputs)
print ('inputs:')
print(inputs)
print ('queries:')
print(queries)

# W_key 是键向量的线性变换层
# keys 的形状也是 [num_tokens, d_out]
keys = sa_v2.W_key(inputs) 
print('keys:')
print(keys)

# 计算注意力分数
# queries @ keys.T 表示每个 query 和每个 key 做点积
# 结果 attn_scores 的形状是 [num_tokens, num_tokens]
# 每一行表示：当前词对所有词的注意力分数
attn_scores = queries @ keys.T
print('attn_scores:')
print(attn_scores)

# 将注意力分数转换为注意力权重
# keys.shape[-1] 表示 key 向量的维度
# 除以 key 维度的平方根，是缩放点积注意力的标准做法
# dim=-1 表示对每一行做 softmax，让每一行的权重和为 1
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

# 打印普通注意力权重矩阵
# 此时每个词仍然可以关注未来词
print('attn_weights:')
print(attn_weights)

#这段代码得到的是一个完整的注意力矩阵：
#词1 可以看 词1、词2、词3、词4、词5、词6
#词2 可以看 词1、词2、词3、词4、词5、词6
#词3 可以看 词1、词2、词3、词4、词5、词6
#...
#这对 GPT 来说不合适，因为 GPT 预测下一个词时不能偷看未来词。

inputs:
tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
queries:
tensor([[ 0.6600, -0.2047],
        [ 0.9091, -0.4471],
        [ 0.8960, -0.4419],
        [ 0.5034, -0.2633],
        [ 0.4088, -0.2232],
        [ 0.6628, -0.3292]], grad_fn=<MmBackward0>)
keys:
tensor([[ 0.3147, -0.4016],
        [-0.0298, -0.4459],
        [-0.0170, -0.4262],
        [-0.1054, -0.2724],
        [ 0.2185,  0.0482],
        [-0.2258, -0.4782]], grad_fn=<MmBackward0>)
attn_scores:
tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  

- The simplest way to mask out future attention weights is by creating a mask via PyTorch's tril function with elements below the main diagonal (including the diagonal itself) set to 1 and above the main diagonal set to 0:

In [66]:
#构造简单的下三角因果掩码
# attn_scores 的形状是 [num_tokens, num_tokens]
# shape[0] 表示输入序列中的词元数量
# 这里 context_length 就是上下文长度，也就是当前序列长度
context_length = attn_scores.shape[0]
print('context_length')
print(context_length)
# 创建一个 context_length × context_length 的全 1 矩阵

# torch.tril 表示保留下三角部分，把上三角部分变成 0
# 得到的 mask_simple 形如：
# 1 0 0 0
# 1 1 0 0
# 1 1 1 0
# 1 1 1 1
mask_simple = torch.tril(torch.ones(context_length, context_length))
# 打印这个下三角掩码矩阵
print(mask_simple)

#这个 mask 的含义是：
#第 1 个词：只能看第 1 个词
#第 2 个词：只能看第 1、2 个词
#第 3 个词：只能看第 1、2、3 个词
#第 4 个词：只能看第 1、2、3、4 个词
#也就是：
#左边和自己可以看，右边未来词不能看。

context_length
6
tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


- Then, we can multiply the attention weights with this mask to zero out the attention scores above the diagonal:

In [ ]:
#用下三角掩码屏蔽未来词
# 将普通注意力权重矩阵和下三角掩码矩阵逐元素相乘
# mask_simple 中为 1 的位置保留原值
# mask_simple 中为 0 的位置变成 0
# 这样就把未来词的注意力权重屏蔽掉了
masked_simple = attn_weights*mask_simple
# 打印屏蔽后的注意力权重矩阵
# 可以看到对角线右上方的位置都变成了 0
print('masked_simple:')
print(masked_simple)

#原来第 2 个词可以关注：词1、词2、词3、词4、词5、词6
#加 mask 后：第 2 个词只能关注：词1、词2

masked_simple
tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


- However, if the mask were applied after softmax, like above, it would disrupt the probability distribution created by softmax
- Softmax ensures that all output values sum to 1
- Masking after softmax would require re-normalizing the outputs to sum to 1 again, which complicates the process and might lead to unintended effects

- To make sure that the rows sum to 1, we can normalize the attention weights as follows:

In [27]:
#重新归一化注意力权重
#前面直接把未来位置清零后，每一行的权重之和可能不再等于 1，所以需要重新归一化。
# 对 masked_simple 的每一行求和
# dim=-1 表示沿着最后一个维度求和，也就是每一行求和
# keepdim=True 表示保留维度，方便后面做广播除法
row_sums = masked_simple.sum(dim=-1, keepdim=True)
# 将每一行中的每个元素都除以这一行的总和
# 这样可以保证每一行的注意力权重重新加起来等于 1
masked_simple_norm = masked_simple / row_sums
# 打印重新归一化后的注意力权重矩阵
# 未来词位置仍然是 0
# 每一行保留下来的权重之和为 1
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


- While we are technically done with coding the causal attention mechanism now, let's briefly look at a more efficient approach to achieve the same as above
- So, instead of zeroing out attention weights above the diagonal and renormalizing the results, we can mask the unnormalized attention scores above the diagonal with negative infinity before they enter the softmax function:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/21.webp" width="450px">

In [28]:
#更高效的因果掩码实现
#这段代码是更推荐的实现方式：不是先 softmax 再清零，而是在 softmax 之前，把未来位置的注意力分数改成 -inf。
# 创建一个上三角矩阵
# torch.triu 表示保留上三角
# diagonal=1 表示从主对角线右上方开始保留
# 主对角线本身不屏蔽，因为当前位置可以看自己
# 得到的 mask 形如：
# 0 1 1 1
# 0 0 1 1
# 0 0 0 1
# 0 0 0 0
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
# mask.bool() 将 0/1 矩阵转换成布尔矩阵
# 1 变成 True，表示这些位置需要被屏蔽
# 0 变成 False，表示这些位置保留
# masked_fill 会把 True 的位置替换成 -inf
# -torch.inf 表示负无穷
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
# 打印屏蔽后的注意力分数矩阵
# 对角线右上方的位置会变成 -inf
print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


- As we can see below, now the attention weights in each row correctly sum to 1 again:

In [ ]:
#这段代码把已经填入 -inf 的注意力分数转换为最终的因果注意力权重。
# 对 masked 注意力分数做缩放
# keys.shape[-1] 是 key 向量的维度
# 除以 key 维度的平方根，可以避免点积值过大导致 softmax 梯度过小
# dim=1 表示对每一行做 softmax
# 因为未来位置已经是 -inf，所以 softmax 后这些位置会变成 0
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)

# 打印最终的因果注意力权重
# 每一行的右上方未来位置都是 0
# 每一行的权重和仍然是 1
print(attn_weights)

#这段代码完成了真正的因果注意力权重计算：
#当前词只能关注自己和之前的词，不能关注后面的词。
#这也是 GPT 生成式模型必须具备的性质。

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


### 3.5.2 Masking additional attention weights with dropout

- In addition, we also apply dropout to reduce overfitting during training
- Dropout can be applied in several places:
  - for example, after computing the attention weights;
  - or after multiplying the attention weights with the value vectors
- Here, we will apply the dropout mask after computing the attention weights because it's more common

- Furthermore, in this specific example, we use a dropout rate of 50%, which means randomly masking out half of the attention weights. (When we train the GPT model later, we will use a lower dropout rate, such as 0.1 or 0.2

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/22.webp" width="400px">

- If we apply a dropout rate of 0.5 (50%), the non-dropped values will be scaled accordingly by a factor of 1/0.5 = 2.

In [ ]:
#构造 dropout 层并测试效果
#这段代码先用一个全 1 矩阵演示 dropout 的效果：随机把一部分元素置 0，保留下来的元素会被放大。
#设置随机种子，保证每次运行时 dropout 随机结果尽量可复现
torch.manual_seed(123)
# 创建一个 dropout 层
# 0.5 表示有 50% 的元素会被随机置为 0
# 注意：dropout 主要用于训练阶段，推理阶段通常关闭
dropout = torch.nn.Dropout(0.5) # dropout rate of 50%
# 创建一个 6×6 的全 1 矩阵
# 这个矩阵只是用来演示 dropout 的行为
example = torch.ones(6, 6) # create a matrix of ones
# 对全 1 矩阵应用 dropout
# 大约一半位置会变成 0
# 没有被置 0 的位置会被放大为 2
# 因为 dropout 会按 1 / (1 - p) 进行缩放
# 当 p=0.5 时，缩放系数就是 2
print(dropout(example))

#dropout 的作用是：
#训练时随机关掉一些连接，防止模型过度依赖某几个固定位置。
#比如 dropout 率是 0.5，就相当于告诉模型：
#这次训练中，你只能随机使用一半的信息。
#这样可以降低过拟合风险。

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [ ]:
# 再次设置随机种子，方便复现实验结果
torch.manual_seed(123)
# 对已经计算好的因果注意力权重应用 dropout
# 一部分注意力权重会被随机置为 0
# 其余保留的权重会被按比例放大
# 这样可以让模型训练时不要总依赖某些固定词元
print(dropout(attn_weights))


#通熟理解就是：
#因果 mask 是固定规则：未来词必须屏蔽。
#dropout 是随机规则：训练时再随机屏蔽一些允许看的词。
#所以它们的区别是：
#因果 mask：为了防止偷看答案
#dropout：为了防止模型过拟合

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


- Note that the resulting dropout outputs may look different depending on your operating system; you can read more about this inconsistency [here on the PyTorch issue tracker](https://github.com/pytorch/pytorch/issues/121595)

### 3.5.3 Implementing a compact causal self-attention class

- Now, we are ready to implement a working implementation of self-attention, including the causal and dropout masks
- One more thing is to implement the code to handle batches consisting of more than one input so that our `CausalAttention` class supports the batch outputs produced by the data loader we implemented in chapter 2
- For simplicity, to simulate such batch input, we duplicate the input text example:

In [ ]:
#构造批量输入 batch
#前面的代码处理的是单个输入序列。真正训练模型时，一般是一批数据一起输入。所以这里用两个相同的 inputs 堆叠成一个 batch，用来测试 CausalAttention 是否支持批量输入。
# torch.stack 会把多个张量沿着一个新维度拼接起来
# 这里把两个相同的 inputs 组成一个 batch
# dim=0 表示在最前面增加 batch 维度
batch = torch.stack((inputs, inputs), dim=0)
print (batch)
# 打印 batch 的形状
# 如果 inputs 的形状是 [6, 3]
# 那么 batch 的形状就是 [2, 6, 3]
# 2 表示 batch 中有 2 个样本
# 6 表示每个样本有 6 个词元
# 3 表示每个词元的嵌入维度是 3
print(batch.shape) # 2 inputs with 6 tokens each, and each token has embedding dimension 3

#原来只有一句话：[6 个词，每个词 3 维]
#现在变成一批：[2 个样本，每个样本 6 个词，每个词 3 维]
#也就是：[batch_size, num_tokens, embedding_dim]

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
torch.Size([2, 6, 3])


In [ ]:
#实现 CausalAttention 类
#这是 3.5 节最重要的代码。它把前面的逻辑封装成一个 PyTorch 模块：
#输入 x
#  ↓
#计算 query、key、value
#  ↓
#计算注意力分数
#  ↓
#用因果 mask 屏蔽未来词
#  ↓
#softmax 得到注意力权重
#  ↓
#dropout 随机屏蔽部分注意力权重
#  ↓
#乘以 values 得到上下文向量
# 定义一个因果注意力模块
# nn.Module 是 PyTorch 中所有神经网络模块的基类
class CausalAttention(nn.Module):

    # 初始化函数
    # d_in：输入嵌入维度
    # d_out：输出嵌入维度
    # context_length：最大上下文长度，也就是最大词元数量
    # dropout：dropout 概率
    # qkv_bias：query/key/value 线性层是否使用偏置项
    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        # 调用父类 nn.Module 的初始化函数
        # 这是 PyTorch 自定义模块的标准写法
        super().__init__()
        # 保存输出维度
        # 后续多头注意力中会用到这个属性
        self.d_out = d_out

        # 定义 query 线性层
        # 作用：把输入 x 映射成 query 向量
        # 输入维度 d_in，输出维度 d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 定义 key 线性层
        # 作用：把输入 x 映射成 key 向量
        # key 用来和 query 做点积，计算注意力分数
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 定义 value 线性层
        # 作用：把输入 x 映射成 value 向量
        # 最终上下文向量是 value 的加权求和
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # 定义 dropout 层
        # 作用：训练时随机屏蔽一部分注意力权重，降低过拟合风险
        self.dropout = nn.Dropout(dropout) # New

        # 注册一个名为 mask 的 buffer
        # buffer 不是可训练参数，不会被优化器更新
        # 但它会随着模型一起保存、加载，并且会跟随模型移动到 CPU/GPU
        # buffer 的名字叫 mask
        # 创建一个上三角矩阵
        # context_length × context_length 表示最大注意力矩阵大小
        # diagonal=1 表示主对角线右上方的位置为 1
        # 这些位置代表未来词，需要屏蔽
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # New

# 前向传播函数
# x 是输入张量
# x 的形状是 [batch_size, num_tokens, d_in]

    def forward(self, x):
        # 读取输入张量的形状
        # b：batch size，一批中有多少个样本
        # num_tokens：当前输入序列有多少个词元
        # d_in：每个词元的输入嵌入维度
        b, num_tokens, d_in = x.shape # New batch dimension b

        # 计算 key 向量
        # keys 的形状是 [batch_size, num_tokens, d_out]
        keys = self.W_key(x)

        # 计算 query 向量
        # queries 的形状是 [batch_size, num_tokens, d_out]
        queries = self.W_query(x)

        # 计算 value 向量
        # values 的形状是 [batch_size, num_tokens, d_out]
        values = self.W_value(x)

# 计算注意力分数

        # keys.transpose(1, 2) 会交换 keys 的第 1 维和第 2 维
        # keys 原形状：[batch_size, num_tokens, d_out]
        # 转置后：[batch_size, d_out, num_tokens]
        # queries @ keys.transpose(1, 2) 得到：
        # [batch_size, num_tokens, num_tokens]
        # 表示每个词对其他所有词的注意力分数
        attn_scores = queries @ keys.transpose(1, 2) # Changed transpose

        # 对注意力分数应用因果掩码
        # self.mask.bool() 将 0/1 mask 转成布尔类型
        # [:num_tokens, :num_tokens] 是为了适配当前输入长度
        # 因为实际输入长度可能小于最大 context_length
        # masked_fill_ 中的下划线表示原地修改
        # 将未来词位置填成 -inf
        attn_scores.masked_fill_(  # New, _ ops are in-place
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  # `:num_tokens` to account for cases where the number of tokens in the batch is smaller than the supported context_size
        
        # 对掩码后的注意力分数做 softmax
        # 除以 keys.shape[-1]**0.5 是缩放操作
        # dim=-1 表示对每一行进行 softmax
        # 未来词位置是 -inf，softmax 后会变成 0
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        # 对注意力权重应用 dropout
        # 训练时随机将部分注意力权重置为 0
        # 推理时 dropout 会自动关闭
        attn_weights = self.dropout(attn_weights) # New

        # 用注意力权重对 value 向量进行加权求和
        # attn_weights 形状：[batch_size, num_tokens, num_tokens]
        # values 形状：[batch_size, num_tokens, d_out]
        # 输出 context_vec 形状：[batch_size, num_tokens, d_out]
        context_vec = attn_weights @ values

        # 返回上下文向量
        # 每个词元都会得到一个新的上下文表示
        return context_vec

#实例化并测试 CausalAttention
# 设置随机种子
# 这样线性层中的随机初始化权重尽量保持可复现
torch.manual_seed(123)

# 获取上下文长度
# batch 的形状是 [batch_size, num_tokens, d_in]
# batch.shape[1] 就是 num_tokens，即每个样本的词元数量
context_length = batch.shape[1]

# 创建一个 CausalAttention 实例
# d_in：输入嵌入维度
# d_out：输出嵌入维度
# context_length：上下文长度
# 0.0：dropout 概率，这里设置为 0，表示不随机丢弃注意力权重
ca = CausalAttention(d_in, d_out, context_length, 0.0)

# 将 batch 输入到因果注意力模块中
# PyTorch 会自动调用 ca.forward(batch)
# 输出 context_vecs 是每个词元对应的上下文向量
context_vecs = ca(batch)

# 打印输出张量的形状
# 预期形状是 [batch_size, num_tokens, d_out]
# 例如 [2, 6, 2]
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

#通俗理解
#输入 batch 形状是：
#[2, 6, 3]
#含义是：
#2 个样本
#每个样本 6 个词元
#每个词元 3 维
#经过 CausalAttention(d_in=3, d_out=2) 后，输出形状变成：
#[2, 6, 2]
#含义是：
#2 个样本
#每个样本 6 个词元
#每个词元被转换成 2 维上下文向量

#把 3.5 节代码连起来看，本质流程是：
#1. 先计算普通注意力分数
#   queries @ keys.T

#2. 再构造因果 mask
#   屏蔽未来词

#3. 把未来位置填成 -inf
#   防止 softmax 给它分配权重

#4. softmax 得到因果注意力权重
#   每一行权重和为 1

#5. 对注意力权重做 dropout
#   降低过拟合风险

#6. 用注意力权重加权 value
#   得到上下文向量

#7. 封装成 CausalAttention 类
#   后面可以直接作为 GPT 模型组件使用

#CausalAttention = SelfAttention + 上三角 mask + dropout。
#普通注意力：所有词都能互相看
#因果注意力：只能看自己和左边，不能看右边
#dropout：训练时再随机遮掉一部分注意力，防止过拟合
#这就是 GPT 生成文本时能够“从左到右逐词预测”的关键机制。

tensor([[[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]],

        [[-0.4519,  0.2216],
         [-0.5874,  0.0058],
         [-0.6300, -0.0632],
         [-0.5675, -0.0843],
         [-0.5526, -0.0981],
         [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


- Note that dropout is only applied during training, not during inference

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/23.webp" width="500px">

## 3.6 Extending single-head attention to multi-head attention

### 3.6.1 Stacking multiple single-head attention layers

- Below is a summary of the self-attention implemented previously (causal and dropout masks not shown for simplicity)

- This is also called single-head attention:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/24.webp" width="400px">

- We simply stack multiple single-head attention modules to obtain a multi-head attention module:

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/25.webp" width="400px">

- The main idea behind multi-head attention is to run the attention mechanism multiple times (in parallel) with different, learned linear projections. This allows the model to jointly attend to information from different representation subspaces at different positions.

In [58]:
#1. MultiHeadAttentionWrapper 类
#   作用：用多个 CausalAttention 模块拼接成多头注意力。

#2. MultiHeadAttentionWrapper 测试代码
#   作用：验证多个头拼接后输出维度如何变化。

#3. MultiHeadAttention 类
#   作用：高效版多头注意力，一次性计算 Q/K/V，再切分为多个头并行计算。

#4. 批量矩阵乘法演示代码
#   作用：帮助理解多头注意力中 4 维张量如何做矩阵乘法。

#5. MultiHeadAttention 测试代码
#   作用：验证高效版多头注意力的输入输出形状。

#需要注意：第 3.6 节的代码依赖前面 3.5 节已经实现的 CausalAttention 类，还依赖 torch、torch.nn as nn 和前面构造好的 batch 输入。

#创建多个 CausalAttention 单头注意力模块
#每个头独立处理同一个输入
#最后把每个头的输出在最后一个维度上拼接起来
#也就是说：
#输入 x
#  ├── head1(x)
#  ├── head2(x)
#  ├── head3(x)
#  └── ...
#最后 torch.cat 拼接
#这种方法直观，但不够高效，因为每个头都单独做一次 Q/K/V 投影。
class MultiHeadAttentionWrapper(nn.Module): # 定义一个多头注意力封装类，继承 PyTorch 的 nn.Module

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):# 初始化函数，接收输入维度、输出维度、上下文长度 接收 dropout 概率、注意力头数量，以及 Q/K/V 是否使用偏置
        super().__init__() # 调用父类 nn.Module 的初始化方法，这是 PyTorch 模块必须做的初始化
        self.heads = nn.ModuleList(# 创建一个 ModuleList，用来保存多个注意力头
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) # 每一个注意力头都是一个 CausalAttention 单头因果注意力模块 # 给每个单头注意力传入相同的配置参数 # 一个 CausalAttention 对象创建结束
             for _ in range(num_heads)]# 根据 num_heads 创建多个 CausalAttention 模块
        )

    def forward(self, x):# 定义前向传播函数，x 是输入张量
        return torch.cat([head(x) for head in self.heads], dim=-1)# 让每个注意力头处理 x，然后在最后一个维度拼接结果


#这段代码用来测试 MultiHeadAttentionWrapper。
#1. 固定随机种子
#2. 设置上下文长度
#3. 设置输入维度和每个头的输出维度
#4. 创建一个包含 2 个头的多头注意力模块
#5. 把 batch 输入进去
#6. 打印输出结果和输出形状
torch.manual_seed(123)# 设置随机种子，保证每次运行时初始化权重尽量一致，方便复现实验结果

context_length = batch.shape[1] # This is the number of tokens # 获取输入序列长度，也就是每个样本中有多少个词元
d_in, d_out = 3, 2 # 设置输入嵌入维度为 3，每个注意力头的输出维度为 2
mha = MultiHeadAttentionWrapper( # 创建一个 MultiHeadAttentionWrapper 多头注意力模块 # 输入维度 3，单头输出维度 2，上下文长度，dropout 为 0，头数为 2
    d_in, d_out, context_length, 0.0, num_heads=2
)

context_vecs = mha(batch)# 将 batch 输入多头注意力模块，得到上下文向量

print(context_vecs)# 打印多头注意力输出的上下文向量
print("context_vecs.shape:", context_vecs.shape)# 打印输出张量的形状，方便检查维度是否符合预期

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


- In the implementation above, the embedding dimension is 4, because we `d_out=2` as the embedding dimension for the key, query, and value vectors as well as the context vector. And since we have 2 attention heads, we have the output embedding dimension 2*2=4

### 3.6.2 Implementing multi-head attention with weight splits

- While the above is an intuitive and fully functional implementation of multi-head attention (wrapping the single-head attention `CausalAttention` implementation from earlier), we can write a stand-alone class called `MultiHeadAttention` to achieve the same

- We don't concatenate single attention heads for this stand-alone `MultiHeadAttention` class
- Instead, we create single W_query, W_key, and W_value weight matrices and then split those into individual matrices for each attention head:

In [ ]:
#高效版 MultiHeadAttention 类
#只创建一组大的 W_query、W_key、W_value
#一次性算出所有头的 Q/K/V
#然后通过 view 和 transpose 把它们切成多个头
#并行计算所有头的注意力
#最后再合并
#书中指出，这种方式比 MultiHeadAttentionWrapper 更高效，因为它只需要一次矩阵乘法来计算键矩阵、查询矩阵和值矩阵，而不是每个头都重复计算。
class MultiHeadAttention(nn.Module):# 定义高效版多头注意力类，继承 PyTorch 的 nn.Module
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):# 初始化函数，d_in 是输入维度，d_out 是总输出维度 # 接收上下文长度、dropout、头数、Q/K/V 是否使用偏置
        super().__init__() # 调用父类 nn.Module 的初始化函数
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads" # 断言：总输出维度 d_out 必须能被头数 num_heads 整除 # 如果不能整除，就抛出错误提示

        self.d_out = d_out # 保存总输出维度，例如 GPT-2 small 中通常是 768
        self.num_heads = num_heads # 保存注意力头数量，例如 GPT-2 small 是 12 个头
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim # 计算每个头的维度，例如 768 // 12 = 64

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # 定义查询 Q 的线性投影层，把输入从 d_in 映射到 d_out
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)# 定义键 K 的线性投影层，把输入从 d_in 映射到 d_out
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)# 定义键 K 的线性投影层，把输入从 d_in 映射到 d_out
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs# 定义输出投影层，用于把多个头合并后的结果再做一次线性变换
        self.dropout = nn.Dropout(dropout)# 定义 dropout 层，用于训练时随机丢弃部分注意力权重，降低过拟合
        self.register_buffer(# 注册一个不会被训练更新、但会随模型移动到 CPU/GPU 的缓冲区
            "mask",# 这个缓冲区名字叫 mask，用来保存因果注意力掩码
            torch.triu(torch.ones(context_length, context_length), # 创建一个上三角矩阵，大小是 context_length × context_length
                       diagonal=1)# diagonal=1 表示主对角线以上的位置为 1，用来屏蔽未来词元
        )# 因果 mask 注册完成

    def forward(self, x):# 定义前向传播函数，x 是输入张量
        b, num_tokens, d_in = x.shape # 读取输入张量形状：b 是 batch size，num_tokens 是词元数量，d_in 是输入维度

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out) # 通过 W_key 线性层，把输入 x 投影成键矩阵 K，形状为 (b, num_tokens, d_out)
        queries = self.W_query(x) # 通过 W_query 线性层，把输入 x 投影成查询矩阵 Q，形状为 (b, num_tokens, d_out)
        values = self.W_value(x) # 通过 W_value 线性层，把输入 x 投影成值矩阵 V，形状为 (b, num_tokens, d_out)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) # 把 K 的 d_out 拆成 num_heads × head_dim
        values = values.view(b, num_tokens, self.num_heads, self.head_dim) # 把 V 的 d_out 拆成 num_heads × head_dim
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)# 把 Q 的 d_out 也拆成 num_heads × head_dim # 新形状为 (b, num_tokens, num_heads, head_dim) # Q 的形状重塑完成

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2) # 交换第 1 维和第 2 维，把 K 变成 (b, num_heads, num_tokens, head_dim)
        queries = queries.transpose(1, 2) # 交换第 1 维和第 2 维，把 Q 变成 (b, num_heads, num_tokens, head_dim)
        values = values.transpose(1, 2) # 交换第 1 维和第 2 维，把 V 变成 (b, num_heads, num_tokens, head_dim)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head # 计算注意力分数 Q @ K^T，得到形状 (b, num_heads, num_tokens, num_tokens)

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens] # 根据当前输入长度截取 mask，并转成布尔类型

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf) # 把未来位置的注意力分数填成负无穷，防止模型看到未来词元
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1) # 对注意力分数做 softmax，得到注意力权重 # 除以 sqrt(head_dim) 做缩放，dim=-1 表示在最后一维归一化
        attn_weights = self.dropout(attn_weights) # 对注意力权重应用 dropout，训练时随机屏蔽一部分注意力连接

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)  # 用注意力权重加权 V，得到上下文向量，并转回 (b, num_tokens, num_heads, head_dim)
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)# 保证张量内存连续，然后重新塑形 # 把 num_heads × head_dim 合并回 d_out，形状变成 (b, num_tokens, d_out) # 多个头的结果合并完成
        context_vec = self.out_proj(context_vec) # optional projection # 通过输出投影层，把多个头合并后的信息再融合一次

        return context_vec # 返回最终的多头注意力上下文向量


#这段代码测试高效版 MultiHeadAttention。
#1. 设置随机种子
#2. 从 batch 中读取 batch_size、context_length、d_in
#3. 设置总输出维度 d_out
#4. 创建高效版多头注意力模块
#5. 执行前向传播
#6. 打印输出结果和输出形状

# 设置随机种子，保证模型权重初始化尽量可复现
torch.manual_seed(123)

# 从 batch 中读取 batch size、上下文长度和输入维度
batch_size, context_length, d_in = batch.shape
d_out = 2 # 设置多头注意力模块的总输出维度为 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2) # 创建高效版多头注意力模块，使用 2 个头，dropout 为 0

context_vecs = mha(batch) # 把 batch 输入到多头注意力模块，得到上下文向量

print(context_vecs) # 打印输出的上下文向量
print("context_vecs.shape:", context_vecs.shape) # 打印输出张量形状

#2：batch 中有 2 个样本
#6：每个样本有 6 个词元
#2：每个词元最终输出 2 维上下文向量

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


- Note that the above is essentially a rewritten version of `MultiHeadAttentionWrapper` that is more efficient
- The resulting output looks a bit different since the random weight initializations differ, but both are fully functional implementations that can be used in the GPT class we will implement in the upcoming chapters
- Note that in addition, we added a linear projection layer (`self.out_proj `) to the `MultiHeadAttention` class above. This is simply a linear transformation that doesn't change the dimensions. It's a standard convention to use such a projection layer in LLM implementation, but it's not strictly necessary (recent research has shown that it can be removed without affecting the modeling performance; see the further reading section at the end of this chapter)


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="400px">

- Note that if you are interested in a compact and efficient implementation of the above, you can also consider the [`torch.nn.MultiheadAttention`](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) class in PyTorch

- Since the above implementation may look a bit complex at first glance, let's look at what happens when executing `attn_scores = queries @ keys.transpose(2, 3)`:

In [61]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],# 第 1 个 batch、第 1 个 head、第 1 个词元的 4 维向量
                    [0.8993, 0.0390, 0.9268, 0.7388],# 第 1 个 batch、第 1 个 head、第 2 个词元的 4 维向量
                    [0.7179, 0.7058, 0.9156, 0.4340]],# 第 1 个 batch、第 1 个 head、第 3 个词元的 4 维向量

                   [[0.0772, 0.3565, 0.1479, 0.5331],# 第 1 个 batch、第 2 个 head、第 1 个词元的 4 维向量
                    [0.4066, 0.2318, 0.4545, 0.9737],# 第 1 个 batch、第 2 个 head、第 2 个词元的 4 维向量
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])# 第 1 个 batch、第 2 个 head、第 3 个词元的 4 维向量

print(a.shape)
print(a @ a.transpose(2, 3))

torch.Size([1, 2, 3, 4])
tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


- In this case, the matrix multiplication implementation in PyTorch will handle the 4-dimensional input tensor so that the matrix multiplication is carried out between the 2 last dimensions (num_tokens, head_dim) and then repeated for the individual heads 

- For instance, the following becomes a more compact way to compute the matrix multiplication for each head separately:

In [ ]:
first_head = a[0, 0, :, :] # 取出第 1 个 batch 中的第 1 个 head，形状是 (3, 4)
first_res = first_head @ first_head.T # 第 1 个 head 内部做矩阵乘法，得到形状 (3, 3)
print("First head:\n", first_res) # 打印第 1 个 head 的矩阵乘法结果

second_head = a[0, 1, :, :] # 取出第 1 个 batch 中的第 2 个 head，形状也是 (3, 4)
second_res = second_head @ second_head.T # 第 2 个 head 内部做矩阵乘法，得到形状 (3, 3)
print("\nSecond head:\n", second_res) # 打印第 2 个 head 的矩阵乘法结果

#1. 多头 = 多个注意力头并行看同一段输入
#2. 每个头都有自己的 Q/K/V 子空间
#3. Wrapper 版靠多个 CausalAttention 拼接
#4. 高效版靠 view + transpose 切分 head
#5. 注意力分数仍然是 Q @ K^T
#6. 因果 mask 仍然用于防止偷看未来
#7. 多头结果最后合并回 d_out
#8. out_proj 用来进一步融合多个头的信息
#

#把一个因果注意力头，扩展成多个并行的因果注意力头，然后把多个头的理解结果融合成最终上下文向量。

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


# Summary and takeaways

- See the [./multihead-attention.ipynb](./multihead-attention.ipynb) code notebook, which is a concise version of the data loader (chapter 2) plus the multi-head attention class that we implemented in this chapter and will need for training the GPT model in upcoming chapters
- You can find the exercise solutions in [./exercise-solutions.ipynb](./exercise-solutions.ipynb)